# Putting It Together, 5D Parallelism and LFM2

## TLDR

You have run data parallelism, sharding, and tensor parallelism on four GPUs.
Real training combines several axes at once, and the hard part is choosing the
combination before you spend a thousand GPU hours. This notebook gives you a
planner that estimates per-GPU memory for any model and parallelism plan, so you
can reason about configurations you cannot physically spin up. Then it walks the
decision framework, the way these axes combine into 5D parallelism, and how LFM2
was actually trained.


## Introduction

Over the course you split a training job five different ways in concept, and you
ran three of them for real on four T4 GPUs. Data parallelism replicated or
sharded the model across workers. Sharding with FSDP and ZeRO split the model
state along the data-parallel dimension. Tensor parallelism split the layers
themselves. Pipeline and expert parallelism, the two that need far more hardware,
you met as concepts.

A production run stacks these. A 2048-GPU job might be 8-way data parallel by
8-way tensor parallel by 4-way pipeline by 8-way expert. Picking those numbers is
an exercise in matching memory and communication to your hardware, and you do it
on paper first. So the centerpiece here is not another training run. It is a
planner that estimates what lands on each GPU for any plan, which is exactly what
a real training team builds before it commits the cluster.

That is also the honest answer to a limitation. We cannot run pipeline or expert
parallelism meaningfully on four T4s. But we can reason about them precisely, and
reasoning is the skill that matters at scale.


## Key concepts used in this notebook

**The memory layer and the throughput layer.** Memory techniques, activation
checkpointing, mixed precision, sharding, and offload, decide whether a model
fits. Throughput techniques, the five parallelism axes, decide how fast it
trains. You combine both.

**5D parallelism.** Data, tensor, pipeline, sequence, and expert parallelism
composed together. The total GPU count is the product of the degrees.

**The bandwidth hierarchy as a decision rule.** Put chatty, latency-sensitive
communication on the fastest link, and communication that overlaps with compute
on the slower network.

**Per-GPU memory estimate.** Parameters plus gradients plus optimizer states plus
activations, each reduced by the parallelism degrees that shard it.

**Mixture of experts.** Many expert sub-networks per layer, only a few active per
token, placed across GPUs with expert parallelism. Capacity grows much faster
than compute.


## What you will learn

- How the memory layer and the throughput layer combine in a real plan
- How to estimate per-GPU memory for any model and parallelism plan
- How to read whether a configuration fits, and which lever to turn if not
- A decision framework for choosing axes, tied to the bandwidth hierarchy
- How 5D parallelism reaches thousands of GPUs
- How LFM2 applies these ideas in practice


## Why Ray on Anyscale for the full stack

| Need | How Ray and Anyscale serve it |
|---|---|
| Run any one axis | `TorchTrainer` with FSDP, DeepSpeed, or DTensor, as you did in 01 to 03 |
| Combine axes | Compose the same primitives, or run Megatron-Core under Ray Train with Megatron-Bridge |
| Survive scale | `FailureConfig` and elastic workers from notebook 04 |
| See the whole run | The persisted Anyscale Train dashboard and on-demand profiling |
| Change the plan | Adjust degrees and `num_workers`, the loop stays the same |

The planner below is framework agnostic. Ray is what turns the plan you choose
into a running job, and Anyscale is what runs it at scale.


## What you ran, and what you reasoned about

| Axis | Splits | In this course |
|---|---|---|
| Data parallel | The batch | Ran in 01, sharded in 02 |
| Sharding (ZeRO, FSDP) | Parameters, gradients, optimizer | Ran in 02 |
| Tensor parallel | Weight matrices within a layer | Ran in 03 |
| Pipeline parallel | Layers into stages | Concept in 03, planned here |
| Expert parallel | Experts of a mixture-of-experts | Concept in 03, planned here |

Two layers cut across these. The memory layer decides whether the model fits,
through activation checkpointing, mixed precision, sharding, and offload. The
throughput layer decides how fast it trains, through the parallelism axes. A real
plan chooses from both at once.


## Cell 1 — A per-GPU memory planner

**What you do.** Define a function that estimates per-GPU memory for a model and a
parallelism plan. It uses the same memory model from notebook 00, then reduces
each component by the degree that shards it.

**What to check.** Each lever shows up as a division. Tensor parallel divides the
layer state, pipeline parallel divides by stages, and the ZeRO stage shards
optimizer, gradients, and parameters across the data-parallel dimension.

**Why it matters.** This is how you reason about a run before you launch it. It is
an estimate, real usage adds communication buffers and fragmentation, but it
captures the levers that decide whether a model fits.


In [ ]:
def plan_memory(params, layers, hidden, seq, micro_batch, gpus, dp, tp, pp=1,
                zero_stage=3, activation_checkpointing=True, gpu_mem_gb=80,
                bytes_param=2, optimizer_bytes=8, act_factor=10):
    # Estimate per-GPU training memory for a parallelism plan.
    # bytes_param=2 is fp16. optimizer_bytes=8 is Adam's two fp32 moments.
    assert dp * tp * pp == gpus, f"dp*tp*pp must equal gpus ({dp}*{tp}*{pp} != {gpus})"
    GB = 1e9

    param_b = params * bytes_param
    grad_b  = params * bytes_param
    opt_b   = params * optimizer_bytes
    act_b   = micro_batch * seq * hidden * layers * bytes_param * act_factor
    if activation_checkpointing:
        act_b *= 0.25                       # recompute instead of storing

    # Tensor parallel splits weights and the activations inside the layer.
    param_b, grad_b, opt_b, act_b = (x / tp for x in (param_b, grad_b, opt_b, act_b))
    # Pipeline parallel splits layers across stages.
    param_b, grad_b, opt_b, act_b = (x / pp for x in (param_b, grad_b, opt_b, act_b))
    # ZeRO and FSDP shard state across the data-parallel dimension.
    if zero_stage >= 1: opt_b  /= dp
    if zero_stage >= 2: grad_b /= dp
    if zero_stage >= 3: param_b /= dp

    parts = {"parameters": param_b, "gradients": grad_b,
             "optimizer": opt_b, "activations": act_b}
    total = sum(parts.values()) / GB
    return parts, total, total <= gpu_mem_gb


def show_plan(label, gpu_mem_gb=80, **kw):
    parts, total, fits = plan_memory(gpu_mem_gb=gpu_mem_gb, **kw)
    print(label)
    for name, b in parts.items():
        print(f"  {name:12s} {b/1e9:8.2f} GB")
    verdict = "FITS" if fits else "DOES NOT FIT"
    print(f"  {'total':12s} {total:8.2f} GB   on {gpu_mem_gb} GB  ->  {verdict}")
    print()


## Cell 2 — Reason about real models

**What you do.** Run the planner on three cases. A 1 billion parameter model on
one GPU with no parallelism, a Llama-3 sized 8 billion model on 8 GPUs, and
Qwen2.5-0.5B on our own four T4s.

**What to check.** The 1B case reproduces the roughly 28 GB from notebook 00. The
8B case fits an 80 GB GPU once you apply tensor parallel and ZeRO-3. The 0.5B case
fits our 16 GB T4s with room to spare, which matches what you actually ran in
notebook 03.

**Why it matters.** The same model either fits or does not depending on the plan.
The planner tells you which, before you launch.


In [ ]:
# 1B on one GPU, no parallelism, no checkpointing. Reproduces notebook 00.
show_plan("1B model, 1 GPU, no sharding, no checkpointing",
          params=1e9, layers=24, hidden=2048, seq=1024, micro_batch=16,
          gpus=1, dp=1, tp=1, zero_stage=0, activation_checkpointing=False, gpu_mem_gb=80)

# Llama-3-8B on 8 GPUs, 4-way tensor parallel by 2-way data parallel, ZeRO-3.
show_plan("8B model, 8 GPUs, tp=4 dp=2 ZeRO-3, checkpointing on",
          params=8e9, layers=32, hidden=4096, seq=4096, micro_batch=1,
          gpus=8, dp=2, tp=4, zero_stage=3, activation_checkpointing=True, gpu_mem_gb=80)

# Qwen2.5-0.5B on our four T4s, the run you did in notebook 03.
show_plan("Qwen2.5-0.5B, 4 T4, tp=2 dp=2 ZeRO-3",
          params=0.5e9, layers=24, hidden=896, seq=512, micro_batch=1,
          gpus=4, dp=2, tp=2, zero_stage=3, activation_checkpointing=True, gpu_mem_gb=16)


## Cell 3 — Scale to thousands of GPUs

**What you do.** Plan a 70 billion parameter model across a large cluster, the
dense part of a flagship run.

**What to check.** With tensor parallel, pipeline parallel, and ZeRO-3 sharding,
per-GPU memory drops to a few GB even for a 70B model. Parallelism is what makes
the impossible fit.

**Why it matters.** This is the regime the lecture's 2048-GPU example lives in.
The planner shows why so much parallelism is needed and how it brings a giant
model down to size on each device.


In [ ]:
# 70B dense, 256 GPUs, tp=8 pp=4 dp=8, ZeRO-3.
show_plan("70B model, 256 GPUs, tp=8 pp=4 dp=8 ZeRO-3",
          params=70e9, layers=80, hidden=8192, seq=4096, micro_batch=1,
          gpus=256, dp=8, tp=8, pp=4, zero_stage=3, activation_checkpointing=True, gpu_mem_gb=80)

# A mixture-of-experts run multiplies capacity with expert parallelism on top.
# The lecture's example, DP8 x TP8 x PP4 x EP8, reaches 2048 GPUs. Each token
# still activates only a few experts, so compute grows far slower than capacity.
print("Add EP=8 expert parallelism on top -> 256 x 8 = 2048 GPUs total")


## A decision framework

The planner tells you whether a plan fits. The bandwidth hierarchy tells you
which plan to try first.

| If | Then reach for | Because |
|---|---|---|
| The model fits but you want more throughput | Data parallel, sharded | Cheapest scaling, communication overlaps compute |
| Optimizer and gradients dominate memory | ZeRO or FSDP sharding | Splits state across data-parallel GPUs |
| A single layer is too wide for one GPU | Tensor parallel, inside a node | One all-reduce per layer, wants NVLink |
| The whole model is too deep to fit | Pipeline parallel, across nodes | Splits layers into stages, tolerates slower links |
| Sequences are extremely long | Sequence or context parallel | Splits the sequence dimension |
| You have a mixture-of-experts model | Expert parallel | Places experts across GPUs, routes with all-to-all |

The recurring rule is to match each kind of communication to the right tier of
the bandwidth hierarchy. Tensor parallel on the fast intra-node link, data and
pipeline parallel across the slower network where their communication can hide
behind compute. That single idea, introduced in notebook 00, drives every choice.


## The course in one map

| Notebook | Ray surface you used | What it gave you |
|---|---|---|
| prerequisite | Ray Data | Streaming, sharded data ingestion |
| 00 | the cluster | Why scale, the memory wall, the five axes |
| 01 | TorchTrainer, prepare_model, get_dataset_shard | One GPU to many, data parallel |
| 02 | fully_shard, deepspeed ZeRO | Sharding the model state |
| 03 | DTensor, AutoTP, device mesh | Tensor plus data, 2D parallelism |
| 04 | FailureConfig, profiler, dashboard | Resilience and observability |
| 05 | the planner | Choosing and combining all the axes |

The thread through all of it is one Ray surface. `TorchTrainer`, `prepare_model`,
`ray.train.report`, `ScalingConfig`, and `FailureConfig` carried unchanged from a
single 4-GPU job to sharded, tensor-parallel, fault-tolerant training. You changed
the config, not the code.


## Conclusion

You built a planner that estimates per-GPU memory for any model and parallelism
plan, reasoned about models from 0.5 billion to 70 billion parameters, and saw
how 5D parallelism reaches thousands of GPUs. You have a decision framework tied
to the bandwidth hierarchy, and a concrete picture of how LFM2 combines these
ideas.

Across the course you took a single-GPU loop and scaled it with Ray Train, fed it
with Ray Data, sharded it with FSDP and DeepSpeed, split it with tensor
parallelism, made it fault tolerant, and made it observable. The same handful of
Ray primitives carried the whole way. That is the case for Ray as the orchestrator
for foundation model training, and for Anyscale as the platform that runs it at
scale.

Thank you for building with us.
